# 07 · Robustness checks and summary of hypotheses

Planned checks (Section VII-B): re-estimate E1–E3 (a) without the 6,117 duplicate records, (b) with alternative buffer radii for the speed analysis, and (c) excluding the COVID-19 years 2020–2021.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
import statsmodels.formula.api as smf
from src import models as M
ex = cfg["robustness"]["exclude_years"]
ch = pd.read_parquet(PROC / "city_hour.parquet"); ch = ch[ch["temp_bin"] != "nan"]
ch_nd = pd.read_parquet(PROC / "city_hour_nodup.parquet"); ch_nd = ch_nd[ch_nd["temp_bin"] != "nan"]
cl = pd.read_parquet(PROC / "collision_level.parquet"); cl = cl[cl["temp_bin"] != "nan"].copy()
for c in ["severe", "automobile", "motorcycle", "passenger", "bicycle", "pedestrian"]:
    cl[c] = cl[c].astype(int)
f_freq = M.formula("collisions", M.WEATHER_TERMS + M.CALENDAR_TERMS)
f_sev = M.formula("severe", M.WEATHER_TERMS + M.CALENDAR_TERMS + ["automobile", "motorcycle", "passenger", "bicycle", "pedestrian"])
KEYS = ["rain", "snow", "vis_lt1", "vis_1to5"]
def key_ratios(res, label):
    t = M.ratio_table(res).loc[KEYS]
    return pd.DataFrame({(label, k): [f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"] for k, r in t.iterrows()}).T

## E1 frequency and severity

In [ ]:
specs = {
    "main": (ch, cl),
    "without duplicates": (ch_nd, cl[~cl["is_duplicate"]]),
    f"excluding {ex[0]}-{ex[-1]}": (ch[~ch["year"].isin(ex)], cl[~cl["year"].isin(ex)]),
}
# Without 2020-2021 the COVID-19 indicator is (almost) constant, so it is dropped from that specification.
no_covid = lambda f: f.replace(" + covid", "")
freq, sev = {}, {}
for name, (c, s) in specs.items():
    ff, fs = (no_covid(f_freq), no_covid(f_sev)) if name.startswith("excluding") else (f_freq, f_sev)
    freq[name] = M.ratio_table(M.fit_nb(ff, c)).loc[KEYS]
    sev[name] = M.ratio_table(smf.logit(fs, s).fit(disp=0, maxiter=200)).loc[KEYS]
fmt = lambda t: t.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
rob_e1 = pd.concat({"frequency IRR": pd.DataFrame({k: fmt(v) for k, v in freq.items()}),
                    "severity OR": pd.DataFrame({k: fmt(v) for k, v in sev.items()})})
rob_e1.to_csv(TAB / "robustness_e1.csv")
rob_e1

## E2 speed–collision association: buffer radius

In [ ]:
rows = []
for r in sorted(set(cfg["spatial"]["sensitivity_buffers_m"] + [cfg["spatial"]["buffer_m"]])):
    st = pd.read_parquet(PROC / f"sign_level_{r}m.parquet")
    st = st[(st["active_days"] > 0) & st["p85_speed"].notna() & st["speed_limit"].notna()].copy()
    st["limit"] = st["speed_limit"].astype(int).astype(str)
    res = M.fit_nb("collisions ~ p85_speed + C(limit)", st, np.log(st["active_days"]))
    t = M.ratio_table(res, exclude_prefixes=("Intercept", "alpha")).loc["p85_speed"]
    rows.append({"buffer_m": r, "collisions": int(st["collisions"].sum()), "IRR per km/h": t["estimate"], "ci_low": t["ci_low"], "ci_high": t["ci_high"]})
rob_e2 = pd.DataFrame(rows)
rob_e2.to_csv(TAB / "robustness_e2_buffers.csv", index=False)
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.errorbar(rob_e2["buffer_m"], rob_e2["IRR per km/h"], yerr=[rob_e2["IRR per km/h"] - rob_e2["ci_low"], rob_e2["ci_high"] - rob_e2["IRR per km/h"]], fmt="o", capsize=3)
ax.axhline(1, color="grey", ls="--", lw=0.8); ax.set_xlabel("buffer radius (m)"); ax.set_ylabel("IRR per km/h of 85th-pct speed")
fig.savefig(FIG / "fig_robustness_buffers.png"); plt.show()
rob_e2.round(4)

## E3 exposure model excluding 2020–2021

In [ ]:
spark = get_spark(cfg); spark.sparkContext.setLogLevel("ERROR")
nh = (spark.read.parquet(str(PROC / "neighbourhood_hour"))
      .select("trips", "cyclist_collisions", "rain", "snow", "vis_lt1", "vis_1to5", "temp_bin", "snow_on_ground",
              "hours_since_precip", "hour", "dow", "month", "year", "holiday", "school_day", "daylight", "covid").toPandas())
spark.stop()
nh = nh[(nh["temp_bin"] != "nan") & ~nh["year"].isin(ex)]
f3 = M.formula("cyclist_collisions", M.WEATHER_TERMS + ["C(hour)", "C(dow)", "C(month)", "C(year)", "holiday", "school_day", "daylight"])
rob_e3 = fmt(M.ratio_table(M.fit_nb(f3, nh, np.log(nh["trips"]))).loc[KEYS]).to_frame(f"per-trip IRR excluding {ex[0]}-{ex[-1]}")
rob_e3.to_csv(TAB / "robustness_e3.csv")
rob_e3

## Summary of hypotheses H1–H4

In [ ]:
parts = []
for h, f in [("H1", "h1.csv"), ("H2", "h2.csv"), ("H3", "h3.csv")]:
    p = TAB / f
    if p.exists():
        t = pd.read_csv(p, index_col=0)
        vcol = [c for c in t.columns if c.startswith("verdict")][0]
        parts.append(pd.DataFrame({"hypothesis": h, "quantity": t.index, "estimate": t["estimate"].round(3),
                                   "95% CI": [f"[{a:.3f}, {b:.3f}]" for a, b in zip(t["ci_low"], t["ci_high"])], "verdict": t[vcol]}))
summary = pd.concat(parts, ignore_index=True)
summary.to_csv(TAB / "hypotheses_summary.csv", index=False)
summary

In [ ]:
# H4 is assessed from E4 (notebook 02): Spark vs others by data size, Parquet vs CSV, and speed-up per core
e4 = pd.read_csv(TAB / "e4_median_results.csv")
pq_vs_csv = e4.pivot_table(index=["tool", "cores", "fraction"], columns="fmt", values="total_s")
print("Parquet faster than CSV in", f"{(pq_vs_csv['parquet'] < pq_vs_csv['csv']).mean():.0%}", "of tool/cores/size combinations")
best = e4.loc[e4.groupby(["tool", "fraction"])["total_s"].idxmin()].pivot(index="fraction", columns="tool", values="total_s")
display(best.round(1))
sp = e4[(e4["fraction"] == 1.0) & (e4["fmt"] == "parquet")].pivot(index="cores", columns="tool", values="speedup")
display(sp.round(2))